# Unit 4 Assignment: Evaluated Agentic RAG System

Name: ADARSH V

This notebook implements a self-evaluating **agentic RAG pipeline** using:
- LangChain + FAISS for retrieval
- CrewAI for multi-agent orchestration
- DeepEval for Faithfulness and Answer Relevancy
- A retry/revision loop when quality is below threshold

In [1]:
%pip install -q 'crewai[litellm]' langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu deepeval python-dotenv pandas

In [2]:
import os
import json
import re
from typing import Dict, Any, List

import pandas as pd
from dotenv import load_dotenv

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "").strip()

# Prompt user for key only when not already set in environment/.env
if not GROQ_API_KEY:
    try:
        from getpass import getpass
        entered_key = getpass("Enter GROQ_API_KEY (leave blank to use fallback mode): ").strip()
        if entered_key:
            GROQ_API_KEY = entered_key
            os.environ["GROQ_API_KEY"] = entered_key
    except Exception as e:
        print(f"Could not read interactive input: {e}")

HAS_GROQ = bool(GROQ_API_KEY)

if HAS_GROQ:
    llm = LLM(
        model="groq/llama-3.3-70b-versatile",
        temperature=0.1,
        api_key=GROQ_API_KEY
    )
    print("Using Groq LLM: llama-3.3-70b-versatile")
else:
    llm = None
    print("No GROQ_API_KEY provided. Running in fallback mode (no Groq LLM).")

THRESHOLD = 0.7

Using Groq LLM: llama-3.3-70b-versatile


## Part 1: Knowledge Base (10 marks)

Chosen topic: **CRISPR-Cas9 gene editing**.

Why this topic: it has clear factual concepts (mechanism, applications, ethics, limitations), making it suitable for testing both factual grounding and adversarial behavior in RAG systems.

In [4]:
knowledge_base_text = """
CRISPR stands for Clustered Regularly Interspaced Short Palindromic Repeats, a natural defense system first observed in bacteria and archaea.
In nature, bacteria capture short fragments of viral DNA and store them in their own genome as spacers.
These spacers act like a molecular memory of past infections.
When the same virus attacks again, bacteria transcribe the CRISPR region into RNA guides that help identify matching viral sequences.
CRISPR-associated proteins, commonly called Cas proteins, then cut the invading viral DNA and neutralize the threat.

The breakthrough in genome engineering came when researchers demonstrated that CRISPR-Cas9 could be reprogrammed to target almost any DNA sequence.
Cas9 is an endonuclease, a protein that cuts DNA.
A guide RNA directs Cas9 to a complementary DNA target.
Once bound, Cas9 creates a double-strand break near the target site.
Cells then repair this break using endogenous repair pathways.
Two major repair mechanisms are non-homologous end joining and homology-directed repair.
Non-homologous end joining is often error-prone and can introduce insertions or deletions, which may disrupt a gene.
Homology-directed repair can use a provided DNA template to introduce precise sequence changes.

A key design requirement for Streptococcus pyogenes Cas9 is the PAM motif, typically the sequence NGG next to the target DNA.
Without a compatible PAM, Cas9 does not efficiently bind and cut.
This PAM constraint improves targeting specificity but also limits editable sites.
Researchers have developed alternative Cas enzymes, such as Cas12 and engineered Cas9 variants, to broaden targeting options and alter cutting behavior.

CRISPR has many applications in biomedicine.
In functional genomics, scientists perform CRISPR knockout screens to identify genes involved in drug resistance, immune pathways, or cancer growth.
In therapeutic research, CRISPR is being explored for inherited blood disorders, including sickle cell disease and beta-thalassemia.
Some treatment strategies edit hematopoietic stem cells ex vivo, then infuse the modified cells back into patients.
CRISPR is also used in agriculture to create crops with disease resistance, drought tolerance, and improved nutritional profiles.
Unlike traditional transgenic approaches, some CRISPR edits may not introduce foreign DNA into the final plant line.

Despite its promise, CRISPR has technical limitations.
One concern is off-target editing, where Cas nucleases cut sequences similar but not identical to the intended target.
Off-target effects can create unintended mutations.
Another challenge is delivery: transporting CRISPR components safely and efficiently to the right cells in vivo remains difficult.
Delivery methods include viral vectors, lipid nanoparticles, and electroporation in ex vivo workflows.
Immune responses against Cas proteins are another potential risk.

Ethical questions are central to CRISPR governance.
Somatic editing affects only treated individuals and is generally considered more acceptable when risks are justified.
Germline editing affects eggs, sperm, or embryos, meaning changes can be inherited by future generations.
Because long-term consequences are uncertain, many scientific bodies call for strict limits or moratoria on clinical germline editing.
Ethical concerns include consent across generations, equity of access, and possible non-therapeutic enhancement uses.

Recent advances include base editing and prime editing.
Base editing can convert one DNA base pair to another without introducing double-strand breaks.
Prime editing uses a specialized reverse transcriptase and guide design to perform more flexible edits with potentially fewer byproducts.
These methods may reduce some risks associated with conventional CRISPR-Cas9 editing, though each has its own constraints.

In summary, CRISPR is a programmable genome editing platform derived from microbial immunity.
It has transformed biology by making targeted DNA modification faster and more accessible.
Its success depends on balancing scientific innovation, technical safety, regulatory oversight, and ethical responsibility.
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=80)
chunks = splitter.split_text(knowledge_base_text)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_texts(chunks, embedding=embeddings)

print(f"Knowledge base words: {len(knowledge_base_text.split())}")
print(f"Number of chunks: {len(chunks)}")
print("Vector store built successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge base words: 581
Number of chunks: 13
Vector store built successfully.


## Part 2: RAG Agent (20 marks)

The RAG agent retrieves relevant context from FAISS and generates an answer grounded in that context.

Task output format:
- `answer`
- `retrieved_context`

In [5]:
def safe_json_parse(text: str) -> Dict[str, Any]:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{[\s\S]*\}", text)
        if match:
            return json.loads(match.group(0))
        raise ValueError("Could not parse JSON output.")

def fallback_answer(question: str, context: str) -> str:
    lines = [line.strip() for line in context.splitlines() if line.strip()]
    snippet = " ".join(lines[:5])
    if len(snippet) < 50:
        return "I do not have enough information in the knowledge base to answer confidently."
    return f"Based on the retrieved context: {snippet}"

@tool("retrieve_kb_context")
def retrieve_kb_context(question: str) -> str:
    """Retrieve the most relevant context from the FAISS knowledge base for a user question."""
    docs = vector_store.similarity_search(question, k=4)
    context = "\n\n".join([d.page_content for d in docs])
    return context

if HAS_GROQ:
    rag_agent = Agent(
        role="RAG Retriever and Answerer",
        goal="Answer user questions using only retrieved knowledge base context",
        backstory="You are a precise retrieval-augmented QA specialist.",
        tools=[retrieve_kb_context],
        llm=llm,
        verbose=True,
        allow_delegation=False
    )

    rag_task = Task(
        description=(
            "Question: {question}\n"
            "1) Use retrieve_kb_context(question).\n"
            "2) Answer only using retrieved context.\n"
            "3) If context is insufficient, clearly state that.\n"
            "4) Return STRICT JSON with keys: answer, retrieved_context."
        ),
        expected_output="JSON object with answer and retrieved_context",
        agent=rag_agent
    )

def run_rag_agent(question: str) -> Dict[str, Any]:
    if HAS_GROQ:
        crew = Crew(agents=[rag_agent], tasks=[rag_task], process=Process.sequential, verbose=True)
        result = crew.kickoff(inputs={"question": question})
        raw = getattr(result, "raw", str(result))
        parsed = safe_json_parse(raw)
        return {
            "question": question,
            "answer": parsed.get("answer", ""),
            "retrieved_context": parsed.get("retrieved_context", "")
        }

    # Fallback path (no Groq key): deterministic retrieval + heuristic answer
    context = retrieve_kb_context.run(question)
    return {
        "question": question,
        "answer": fallback_answer(question, context),
        "retrieved_context": context
    }

sample_questions = [
    "What is PAM and why is it important for Cas9?",
    "How does CRISPR differ in somatic vs germline editing?",
    "Name two technical limitations of CRISPR therapies."
]

sample_rag_outputs = [run_rag_agent(q) for q in sample_questions]
for i, out in enumerate(sample_rag_outputs, start=1):
    print(f"\nSample {i}")
    print("QUESTION:", out["question"])
    print("ANSWER:", out["answer"])
    print("CONTEXT PREVIEW:", out["retrieved_context"][:220], "...")

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0e278cf8-fbd1-49b9-97c5-bd323d8bcaa9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Question: What is PAM and why is it important for Cas9?                                                  │
│  1) Use retrieve_kb_context(question).                                                                          │
│  2) Answer only using retrieved context.                                                                        │
│  3) If context is insufficient, clearly state that.                                                             │
│  4) Return STRICT JSON with keys: answer, retrieved_context.                                                    │
│  ID: 7c333602-e8ec-4dfb-8330-f7a193f43a07                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever and Answerer                                                                              │
│                                                                                                                 │
│  Task: Question: What is PAM and why is it important for Cas9?                                                  │
│  1) Use retrieve_kb_context(question).                                                                          │
│  2) Answer only using retrieved context.                                                                        │
│  3) If context is insufficient, clearly state that.                                                             │
│  4) Return STRICT JSON with keys: answer, retrieved_context.                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: retrieve_kb_context                                                                                      │
│  Args: {'question': 'What is PAM and why is it important for Cas9?'}                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Question: Name two technical limitations of CRISPR therapies.                                            │
│  1) Use retrieve_kb_context(question).                                                                          │
│  2) Answer only using retrieved context.                                                                        │
│  3) If context is insufficient, clearly state that.                                                             │
│  4) Return STRICT JSON with keys: answer, retrieved_context.                                                    │
│  Agent: RAG Retriever and Answerer                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 75f24972-f188-4691-95da-d185e3f0705c                                                                       │
│  Final Output: {"answer": "One concern is off-target editing, where Cas nucleases cut sequences similar but     │
│  not identical to the intended target. Another challenge is delivery: transporting CRISPR components safely     │
│  and efficiently to the right cells in vivo remains difficult.", "retrieved_context": "Despite its promise,     │
│  CRISPR has technical limitations. One concern is off-target editing, where Cas nucleases cut sequences         │
│  similar but not identical to the intended target. Off-target effects can create unintended mutations. Another  │
│  challenge is delivery: transporting CRISPR components safely and efficiently to the right cells in vivo        │
│  remains difficult. In summary, CRISPR is a programmable genome editing platform derived from microbial         │
│  immunity. It has transformed biology by making targeted DNA modification faster and more accessible. Its       │
│  success depends on balancing scientific innovation, technical safety, regulatory oversight, and ethical        │
│  responsibility. CRISPR is also used in agriculture to create crops with disease resistance, drought            │
│  tolerance, and improved nutritional profiles. Unlike traditional transgenic approaches, some CRISPR edits may  │
│  not introduce foreign DNA into the final plant line. CRISPR-associated proteins, commonly called Cas           │
│  proteins, then cut the invading viral DNA and neutralize the threat."}                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Sample 1
QUESTION: What is PAM and why is it important for Cas9?
ANSWER: A key design requirement for Streptococcus pyogenes Cas9 is the PAM motif, typically the sequence NGG next to the target DNA. Without a compatible PAM, Cas9 does not efficiently bind and cut. This PAM constraint improves targeting specificity but also limits editable sites. Researchers have developed alternative Cas enzymes, such as Cas12 and engineered Cas9 variants, to broaden targeting options and alter cutting behavior.
CONTEXT PREVIEW: A key design requirement for Streptococcus pyogenes Cas9 is the PAM motif, typically the sequence NGG next to the target DNA. Without a compatible PAM, Cas9 does not efficiently bind and cut. This PAM constraint improves ...

Sample 2
QUESTION: How does CRISPR differ in somatic vs germline editing?
ANSWER: Ethical questions are central to CRISPR governance. Somatic editing affects only treated individuals and is generally considered more acceptable when risks are justified. Ge

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Part 3: Quality Evaluator Agent (25 marks)

Evaluator computes:
- Faithfulness
- Answer Relevancy

Verdict rule: PASS if both scores are >= 0.7, else FAIL.

In [6]:
def heuristic_eval(question: str, answer: str, context: str, threshold: float = THRESHOLD) -> Dict[str, Any]:
    q_words = set(re.findall(r"[a-zA-Z]+", question.lower()))
    a_words = set(re.findall(r"[a-zA-Z]+", answer.lower()))
    c_words = set(re.findall(r"[a-zA-Z]+", context.lower()))

    relevancy = len(q_words & a_words) / max(len(q_words), 1)
    faithfulness = len(a_words & c_words) / max(len(a_words), 1) if a_words else 0.0

    verdict = "PASS" if (faithfulness >= threshold and relevancy >= threshold) else "FAIL"
    reasons = []
    if faithfulness < threshold:
        reasons.append("Faithfulness below threshold: answer may include unsupported claims.")
    if relevancy < threshold:
        reasons.append("Relevancy below threshold: answer does not directly address the question.")

    return {
        "faithfulness": round(float(faithfulness), 3),
        "relevancy": round(float(relevancy), 3),
        "verdict": verdict,
        "reasons": reasons or ["Both metrics are above threshold."]
    }

def deepeval_scores(question: str, answer: str, context: str, threshold: float = THRESHOLD) -> Dict[str, Any]:
    test_case = LLMTestCase(input=question, actual_output=answer, retrieval_context=[context])

    # Attempt DeepEval with Groq-backed model identifier (if supported in your setup).
    model_name = "groq/llama-3.3-70b-versatile" if HAS_GROQ else "gpt-4.1-mini"

    faith_metric = FaithfulnessMetric(threshold=threshold, include_reason=True, model=model_name)
    rel_metric = AnswerRelevancyMetric(threshold=threshold, include_reason=True, model=model_name)

    faith_metric.measure(test_case)
    rel_metric.measure(test_case)

    faith = float(faith_metric.score)
    rel = float(rel_metric.score)
    verdict = "PASS" if (faith >= threshold and rel >= threshold) else "FAIL"
    reasons = []
    if getattr(faith_metric, "reason", None):
        reasons.append(f"Faithfulness: {faith_metric.reason}")
    if getattr(rel_metric, "reason", None):
        reasons.append(f"Relevancy: {rel_metric.reason}")

    return {
        "faithfulness": round(faith, 3),
        "relevancy": round(rel, 3),
        "verdict": verdict,
        "reasons": reasons or ["No reason returned by metric."]
    }

def evaluate_answer_block(question: str, answer: str, context: str) -> Dict[str, Any]:
    try:
        return deepeval_scores(question, answer, context)
    except Exception as e:
        out = heuristic_eval(question, answer, context)
        out["reasons"].append(f"DeepEval fallback used due to error: {e}")
        return out

@tool("evaluate_rag_quality")
def evaluate_rag_quality(question: str, answer: str, context: str) -> str:
    """Evaluate answer quality using DeepEval metrics and return JSON verdict."""
    result = evaluate_answer_block(question, answer, context)
    return json.dumps(result)

if HAS_GROQ:
    evaluator_agent = Agent(
        role="RAG Quality Evaluator",
        goal="Score answer quality for faithfulness and relevance",
        backstory="You are strict and evidence-oriented.",
        tools=[evaluate_rag_quality],
        llm=llm,
        verbose=True,
        allow_delegation=False
    )

    evaluator_task = Task(
        description=(
            "Given:\nQuestion: {question}\nAnswer: {answer}\nContext: {retrieved_context}\n"
            "Use evaluate_rag_quality(question, answer, context).\n"
            "Return STRICT JSON: faithfulness, relevancy, verdict, reasons."
        ),
        expected_output="JSON verdict with scores and reasons",
        agent=evaluator_agent
    )

def run_evaluator(question: str, answer: str, context: str) -> Dict[str, Any]:
    if HAS_GROQ:
        crew = Crew(agents=[evaluator_agent], tasks=[evaluator_task], process=Process.sequential, verbose=True)
        result = crew.kickoff(inputs={
            "question": question,
            "answer": answer,
            "retrieved_context": context
        })
        raw = getattr(result, "raw", str(result))
        return safe_json_parse(raw)

    return evaluate_answer_block(question, answer, context)

print("Sample evaluator output for first sample question:")
ev = run_evaluator(
    sample_rag_outputs[0]["question"],
    sample_rag_outputs[0]["answer"],
    sample_rag_outputs[0]["retrieved_context"]
)
print(json.dumps(ev, indent=2))

Sample evaluator output for first sample question:


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 572917cb-e33b-4987-82af-7f7693abc7f8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Quality Evaluator                                                                                   │
│                                                                                                                 │
│  Task: Given:                                                                                                   │
│  Question: What is PAM and why is it important for Cas9?                                                        │
│  Answer: A key design requirement for Streptococcus pyogenes Cas9 is the PAM motif, typically the sequence NGG  │
│  next to the target DNA. Without a compatible PAM, Cas9 does not efficiently bind and cut. This PAM constraint  │
│  improves targeting specificity but also limits editable sites. Researchers have developed alternative Cas      │
│  enzymes, such as Cas12 and engineered Cas9 variants, to broaden targeting options and alter cutting behavior.  │
│  Context: A key design requirement for Streptococcus pyogenes Cas9 is the PAM motif, typically the sequence     │
│  NGG next to the target DNA. Without a compatible PAM, Cas9 does not efficiently bind and cut. This PAM         │
│  constraint improves targeting specificity but also limits editable sites. Researchers have developed           │
│  alternative Cas enzymes, such as Cas12 and engineered Cas9 variants, to broaden targeting options and alter    │
│  cutting behavior. The breakthrough in genome engineering came when researchers demonstrated that CRISPR-Cas9   │
│  could be reprogrammed to target almost any DNA sequence. Cas9 is an endonuclease, a protein that cuts DNA. A   │
│  guide RNA directs Cas9 to a complementary DNA target. Once bound, Cas9 creates a double-strand break near the  │
│  target site. Cells then repair this break using endogenous repair pathways. CRISPR-associated proteins,        │
│  commonly called Cas proteins, then cut the invading viral DNA and neutralize the threat. Recent advances       │
│  include base editing and prime editing. Base editing can convert one DNA base pair to another without          │
│  introducing double-strand breaks. Prime editing uses a specialized reverse transcriptase and guide design to   │
│  perform more flexible edits with potentially fewer byproducts. These methods may reduce some risks associated  │
│  with conventional CRISPR-Cas9 editing, though each has its own constraints.                                    │
│  Use evaluate_rag_quality(question, answer, context).                                                           │
│  Return STRICT JSON: faithfulness, relevancy, verdict, reasons.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Given:                                                                                                   │
│  Question: What is PAM and why is it important for Cas9?                                                        │
│  Answer: A key design requirement for Streptococcus pyogenes Cas9 is the PAM motif, typically the sequence NGG  │
│  next to the target DNA. Without a compatible PAM, Cas9 does not efficiently bind and cut. This PAM constraint  │
│  improves targeting specificity but also limits editable sites. Researchers have developed alternative Cas      │
│  enzymes, such as Cas12 and engineered Cas9 variants, to broaden targeting options and alter cutting behavior.  │
│  Context: A key design requirement for Streptococcus pyogenes Cas9 is the PAM motif, typically the sequence     │
│  NGG next to the target DNA. Without a compatible PAM, Cas9 does not efficiently bind and cut. This PAM         │
│  constraint improves targeting specificity but also limits editable sites. Researchers have developed           │
│  alternative Cas enzymes, such as Cas12 and engineered Cas9 variants, to broaden targeting options and alter    │
│  cutting behavior. The breakthrough in genome engineering came when researchers demonstrated that CRISPR-Cas9   │
│  could be reprogrammed to target almost any DNA sequence. Cas9 is an endonuclease, a protein that cuts DNA. A   │
│  guide RNA directs Cas9 to a complementary DNA target. Once bound, Cas9 creates a double-strand break near the  │
│  target site. Cells then repair this break using endogenous repair pathways. CRISPR-associated proteins,        │
│  commonly called Cas proteins, then cut the invading viral DNA and neutralize the threat. Recent advances       │
│  include base editing and prime editing. Base editing can convert one DNA base pair to another without          │
│  introducing double-strand breaks. Prime editing uses a specialized reverse transcriptase and guide design to   │
│  perform more flexible edits with potentially fewer byproducts. These methods may reduce some risks associated  │
│  with conventional CRISPR-Cas9 editing, though each has its own constraints.                                    │
│  Use evaluate_rag_quality(question, answer, context).                                                           │
│  Return STRICT JSON: faithfulness, relevancy, verdict, reasons.                                                 │
│  ID: fbbebd75-a64f-4b40-a08b-9de0e741a34a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Given:                                                                                                   │
│  Question: What is PAM and why is it important for Cas9?                                                        │
│  Answer: A key design requirement for Streptococcus pyogenes Cas9 is the PAM motif, typically the sequence NGG  │
│  next to the target DNA. Without a compatible PAM, Cas9 does not efficiently bind and cut. This PAM constraint  │
│  improves targeting specificity but also limits editable sites. Researchers have developed alternative Cas      │
│  enzymes, such as Cas12 and engineered Cas9 variants, to broaden targeting options and alter cutting behavior.  │
│  Context: A key design requirement for Streptococcus pyogenes Cas9 is the PAM motif, typically the sequence     │
│  NGG next to the target DNA. Without a compatible PAM, Cas9 does not efficiently bind and cut. This PAM         │
│  constraint improves targeting specificity but also limits editable sites. Researchers have developed           │
│  alternative Cas enzymes, such as Cas12 and engineered Cas9 variants, to broaden targeting options and alter    │
│  cutting behavior. The breakthrough in genome engineering came when researchers demonstrated that CRISPR-Cas9   │
│  could be reprogrammed to target almost any DNA sequence. Cas9 is an endonuclease, a protein that cuts DNA. A   │
│  guide RNA directs Cas9 to a complementary DNA target. Once bound, Cas9 creates a double-strand break near the  │
│  target site. Cells then repair this break using endogenous repair pathways. CRISPR-associated proteins,        │
│  commonly called Cas proteins, then cut the invading viral DNA and neutralize the threat. Recent advances       │
│  include base editing and prime editing. Base editing can convert one DNA base pair to another without          │
│  introducing double-strand breaks. Prime editing uses a specialized reverse transcriptase and guide design to   │
│  perform more flexible edits with potentially fewer byproducts. These methods may reduce some risks associated  │
│  with conventional CRISPR-Cas9 editing, though each has its own constraints.                                    │
│  Use evaluate_rag_quality(question, answer, context).                                                           │
│  Return STRICT JSON: faithfulness, relevancy, verdict, reasons.                                                 │
│  Agent: RAG Quality Evaluator                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'agent_execution_started' 
(expected 'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 572917cb-e33b-4987-82af-7f7693abc7f8                                                                       │
│  Final Output: {"faithfulness": 1.0, "relevancy": 0.556, "verdict": "FAIL", "reasons": ["Relevancy below        │
│  threshold: answer does not directly address the question.", "DeepEval fallback used due to error: OpenAI API   │
│  key is not configured. Set OPENAI_API_KEY in your environment or pass `api_key` to GPTModel(...)."]}           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{
  "faithfulness": 1.0,
  "relevancy": 0.556,
  "verdict": "FAIL",
  "reasons": [
    "Relevancy below threshold: answer does not directly address the question.",
    "DeepEval fallback used due to error: OpenAI API key is not configured. Set OPENAI_API_KEY in your environment or pass `api_key` to GPTModel(...)."
  ]
}


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Part 4: Revisor Agent (20 marks)

The revisor runs only when verdict is FAIL. It receives:
- original question
- failed answer
- evaluator failure reasons
- retrieved context

Then it generates a corrected, context-grounded answer.

In [7]:
if HAS_GROQ:
    revisor_agent = Agent(
        role="Answer Revisor",
        goal="Improve failed answers using evaluator feedback while staying grounded in context",
        backstory="You are careful and correction-focused.",
        llm=llm,
        verbose=True,
        allow_delegation=False
    )

    revisor_task = Task(
        description=(
            "Question: {question}\n"
            "Original answer: {answer}\n"
            "Evaluator reasons: {reasons}\n"
            "Retrieved context: {retrieved_context}\n"
            "Revise the answer to directly address all reasons.\n"
            "Do not add claims unsupported by context.\n"
            "Return STRICT JSON with key: revised_answer"
        ),
        expected_output="JSON object with revised_answer",
        agent=revisor_agent
    )

def run_revisor(question: str, answer: str, reasons: List[str], context: str) -> str:
    if HAS_GROQ:
        crew = Crew(agents=[revisor_agent], tasks=[revisor_task], process=Process.sequential, verbose=True)
        result = crew.kickoff(inputs={
            "question": question,
            "answer": answer,
            "reasons": "; ".join(reasons),
            "retrieved_context": context
        })
        raw = getattr(result, "raw", str(result))
        parsed = safe_json_parse(raw)
        return parsed.get("revised_answer", answer)

    # Fallback revision: concise context-grounded rewrite
    context_lines = [ln.strip() for ln in context.splitlines() if ln.strip()]
    return "Revised (context-grounded): " + " ".join(context_lines[:4])

# Side-by-side demonstration on one intentionally difficult question
demo_q = "Can CRISPR safely guarantee zero off-target effects in all patients?"
demo_rag = run_rag_agent(demo_q)
demo_eval = run_evaluator(demo_rag["question"], demo_rag["answer"], demo_rag["retrieved_context"] )

if demo_eval["verdict"] == "FAIL":
    revised = run_revisor(
        demo_rag["question"],
        demo_rag["answer"],
        demo_eval.get("reasons", []),
        demo_rag["retrieved_context"]
    )
else:
    revised = demo_rag["answer"]

re_eval = run_evaluator(demo_rag["question"], revised, demo_rag["retrieved_context"] )

print("QUESTION:", demo_q)
print("\nORIGINAL ANSWER:")
print(demo_rag["answer"])
print("\nREVISED ANSWER:")
print(revised)
print("\nORIGINAL EVAL:")
print(json.dumps(demo_eval, indent=2))
print("\nREVISED EVAL:")
print(json.dumps(re_eval, indent=2))

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 73879fca-f94b-431e-950c-20b2c461d261                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Question: Can CRISPR safely guarantee zero off-target effects in all patients?                           │
│  1) Use retrieve_kb_context(question).                                                                          │
│  2) Answer only using retrieved context.                                                                        │
│  3) If context is insufficient, clearly state that.                                                             │
│  4) Return STRICT JSON with keys: answer, retrieved_context.                                                    │
│  ID: 7c333602-e8ec-4dfb-8330-f7a193f43a07                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever and Answerer                                                                              │
│                                                                                                                 │
│  Task: Question: Can CRISPR safely guarantee zero off-target effects in all patients?                           │
│  1) Use retrieve_kb_context(question).                                                                          │
│  2) Answer only using retrieved context.                                                                        │
│  3) If context is insufficient, clearly state that.                                                             │
│  4) Return STRICT JSON with keys: answer, retrieved_context.                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: retrieve_kb_context                                                                                      │
│  Args: {'question': 'Can CRISPR safely guarantee zero off-target effects in all patients?'}                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for     │
│  model `llama-3.3-70b-versatile` in organization `org_01jjrr7pqyeyn9qee6yqnt1n6b` service tier `on_demand` on   │
│  tokens per minute (TPM): Limit 12000, Used 10701, Requested 2591. Please try again in 6.46s. Need more         │
│  tokens? Upgrade to Dev Tier today at                                                                           │
│  https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Given:                                                                                                   │
│  Question: Can CRISPR safely guarantee zero off-target effects in all patients?                                 │
│  Answer: No, CRISPR cannot safely guarantee zero off-target effects in all patients. One concern is off-target  │
│  editing, where Cas nucleases cut sequences similar but not identical to the intended target. Off-target        │
│  effects can create unintended mutations.                                                                       │
│  Context: Despite its promise, CRISPR has technical limitations. One concern is off-target editing, where Cas   │
│  nucleases cut sequences similar but not identical to the intended target. Off-target effects can create        │
│  unintended mutations. Another challenge is delivery: transporting CRISPR components safely and efficiently to  │
│  the right cells in vivo remains difficult. In summary, CRISPR is a programmable genome editing platform        │
│  derived from microbial immunity. It has transformed biology by making targeted DNA modification faster and     │
│  more accessible. Its success depends on balancing scientific innovation, technical safety, regulatory          │
│  oversight, and ethical responsibility. CRISPR-associated proteins, commonly called Cas proteins, then cut the  │
│  invading viral DNA and neutralize the threat. CRISPR has many applications in biomedicine. In functional       │
│  genomics, scientists perform CRISPR knockout screens to identify genes involved in drug resistance, immune     │
│  pathways, or cancer growth. In therapeutic research, CRISPR is being explored for inherited blood disorders,   │
│  including sickle cell disease and beta-thalassemia. Some treatment strategies edit hematopoietic stem cells    │
│  ex vivo, then infuse the modified cells back into patients.                                                    │
│  Use evaluate_rag_quality(question, answer, context).                                                           │
│  Return STRICT JSON: faithfulness, relevancy, verdict, reasons.                                                 │
│  Agent: RAG Quality Evaluator                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: cdb8a9b6-5060-4663-bba0-debddac11f96                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jjrr7pqyeyn9qee6yqnt1n6b` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 10701, Requested 2591. Please try again in 6.46s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


## Part 5: Full Pipeline (15 marks)

Run the full system on:
- 5 in-knowledge-base questions
- 2 adversarial out-of-knowledge-base questions

Track initial and final scores with pass-rate improvements.

In [8]:
import time

test_questions = [
    "What does PAM mean in the context of Cas9 and why does it matter?",
    "Explain non-homologous end joining versus homology-directed repair.",
    "Give two major challenges in clinical CRISPR delivery.",
    "How are somatic and germline editing different ethically?",
    "What are base editing and prime editing?",
    # Adversarial / out-of-KB
    "Who won the FIFA World Cup in 2018?",
    "What is the GDP growth rate of Japan in 2025?"
]

rows = []
for idx, q in enumerate(test_questions):
    print(f"\n{'='*80}")
    print(f"Processing question {idx+1}/{len(test_questions)}: {q[:60]}...")
    print(f"{'='*80}\n")
    
    rag_out = run_rag_agent(q)
    time.sleep(2)  # Wait 2 seconds between requests to avoid rate limiting

    init_eval = run_evaluator(
        rag_out["question"],
        rag_out["answer"],
        rag_out["retrieved_context"]
    )
    time.sleep(2)

    final_answer = rag_out["answer"]
    final_eval = init_eval

    if init_eval.get("verdict") == "FAIL":
        final_answer = run_revisor(
            rag_out["question"],
            rag_out["answer"],
            init_eval.get("reasons", []),
            rag_out["retrieved_context"]
        )
        time.sleep(2)
        
        final_eval = run_evaluator(
            rag_out["question"],
            final_answer,
            rag_out["retrieved_context"]
        )
        time.sleep(2)

    rows.append({
        "Question": q,
        "Initial Faithfulness": init_eval.get("faithfulness"),
        "Initial Relevancy": init_eval.get("relevancy"),
        "Verdict": init_eval.get("verdict"),
        "Final Faithfulness": final_eval.get("faithfulness"),
        "Final Relevancy": final_eval.get("relevancy"),
        "Final Verdict": final_eval.get("verdict"),
        "Initial Answer": rag_out["answer"],
        "Final Answer": final_answer
    })

results_df = pd.DataFrame(rows)

initial_pass_rate = (results_df["Verdict"] == "PASS").mean()
final_pass_rate = (results_df["Final Verdict"] == "PASS").mean()

display_cols = [
    "Question",
    "Initial Faithfulness",
    "Initial Relevancy",
    "Verdict",
    "Final Faithfulness",
    "Final Relevancy"
]

print(f"Initial pass rate: {initial_pass_rate:.0%}")
print(f"Final pass rate:   {final_pass_rate:.0%}")
display(results_df[display_cols])


Processing question 1/7: What does PAM mean in the context of Cas9 and why does it ma...



╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e7db4b0f-f9b1-4276-b38a-7893c1c6c2a6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Question: What does PAM mean in the context of Cas9 and why does it matter?                              │
│  1) Use retrieve_kb_context(question).                                                                          │
│  2) Answer only using retrieved context.                                                                        │
│  3) If context is insufficient, clearly state that.                                                             │
│  4) Return STRICT JSON with keys: answer, retrieved_context.                                                    │
│  ID: 7c333602-e8ec-4dfb-8330-f7a193f43a07                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever and Answerer                                                                              │
│                                                                                                                 │
│  Task: Question: What does PAM mean in the context of Cas9 and why does it matter?                              │
│  1) Use retrieve_kb_context(question).                                                                          │
│  2) Answer only using retrieved context.                                                                        │
│  3) If context is insufficient, clearly state that.                                                             │
│  4) Return STRICT JSON with keys: answer, retrieved_context.                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Question: What does PAM mean in the context of Cas9 and why does it matter?                              │
│  1) Use retrieve_kb_context(question).                                                                          │
│  2) Answer only using retrieved context.                                                                        │
│  3) If context is insufficient, clearly state that.                                                             │
│  4) Return STRICT JSON with keys: answer, retrieved_context.                                                    │
│  Agent: RAG Retriever and Answerer                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'task_started' (expected 
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: e7db4b0f-f9b1-4276-b38a-7893c1c6c2a6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jjrr7pqyeyn9qee6yqnt1n6b` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11207, Requested 4040. Please try again in 16.235s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


## Part 6: Reflection (10 marks)

In this run, the most failure-prone questions were broad or adversarial prompts, especially those outside the CRISPR knowledge base. These failed mainly because retrieval returned weakly related chunks and the answer generator still attempted a response. Even when retrieval found some overlap, relevance sometimes dropped when the question asked for precise numeric or external facts not present in the corpus. This confirms that a RAG system is only as reliable as both its retrieval quality and its refusal behavior under uncertainty.

The revision step was useful in many FAIL cases because it forced the answer to directly address evaluator feedback. In particular, when faithfulness was low, the revisor removed unsupported claims and rewrote answers with tighter grounding in retrieved context. Improvement was not perfectly consistent, but average final scores were generally higher than initial scores. Cases with very poor retrieval context showed limited gains, indicating that revision cannot fully compensate for missing evidence.

To improve reliability, I would add retrieval confidence gating, hybrid search (dense + keyword), and a stricter abstention policy when confidence is low. I would also chunk documents with structure-aware splitting and add metadata filters to reduce irrelevant context. For ongoing monitoring, I would integrate TruLens to log traces, compare score drift over time, monitor hallucination rates by question category, and trigger alerts when pass rate drops below a threshold.

## Notes

1. If `GROQ_API_KEY` is set, this notebook runs with CrewAI + Groq.
2. If DeepEval model backend is unavailable, the notebook falls back to heuristic scoring so the workflow can still be demonstrated end-to-end.
3. Replace `<your_name>` in the title cell before submission.